# Document 对象模型

对标文档：[LangChain > Core Components > Document](https://python.langchain.com/docs/concepts/documents/)

In [1]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_learning.llm import get_llm

llm = get_llm()

**术语：**
- **Document** = LangChain 中表示文本数据的基本单元，包含 `page_content`（文本内容）和 `metadata`（元数据）

```mermaid
%%{init: {'theme': 'dark'}}%%
graph LR
    A[Document] --> B[page_content 文本内容]
    A --> C[metadata 元数据字典]
    B --> D[文档正文]
    C --> E[来源、页码、日期...]
```

### 1. 创建 Document

Document 是 RAG 系统的数据基础，贯穿文档加载、分割、向量化、检索全流程。

In [2]:
# 创建 Document
doc = Document(
    page_content="LangChain 是一个用于构建 LLM 应用的框架。",
    metadata={"source": "wiki", "language": "zh", "id": 1}
)

print(f"内容: {doc.page_content}")
print(f"元数据: {doc.metadata}")
print(f"类型: {doc.type}")

内容: LangChain 是一个用于构建 LLM 应用的框架。
元数据: {'source': 'wiki', 'language': 'zh', 'id': 1}
类型: Document


In [3]:
# 批量创建 Document
docs = [
    Document("Python 是一种动态类型语言", metadata={"topic": "python"}),
    Document("Rust 注重内存安全", metadata={"topic": "rust"}),
    Document("Go 语言并发模型基于 goroutine", metadata={"topic": "go"}),
]

for d in docs:
    print(f"[{d.metadata['topic']}] {d.page_content}")

[python] Python 是一种动态类型语言
[rust] Rust 注重内存安全
[go] Go 语言并发模型基于 goroutine


**术语：**
- **page_content** = Document 的主要文本内容
- **metadata** = 附加信息的字典，如来源、日期、分类等，用于过滤和追踪
- **type** = Document 类型标识（固定为 `"document"`）

`metadata` 在检索阶段特别有用——可以根据元数据过滤文档，或在结果中携带来源信息用于追溯。

### 2. 操作 Document

Document 是可变对象，可以直接修改或通过列表操作批量处理。

In [4]:
# 修改元数据
doc = Document("内容管理系统是一种软件", metadata={"type": "note"})
doc.metadata["author"] = "Alice"
doc.metadata["created_at"] = "2025-01-01"

# 按元数据过滤
all_docs = [
    Document("Python 教程", metadata={"lang": "python", "level": "beginner"}),
    Document("Rust 教程", metadata={"lang": "rust", "level": "intermediate"}),
    Document("Python API", metadata={"lang": "python", "level": "advanced"}),
    Document("Go 基础", metadata={"lang": "go", "level": "beginner"}),
]

python_docs = [d for d in all_docs if d.metadata.get("lang") == "python"]
beginner_docs = [d for d in all_docs if d.metadata.get("level") == "beginner"]

print("Python 相关:")
for d in python_docs:
    print(f"  - {d.page_content}")
print("\n入门级别:")
for d in beginner_docs:
    print(f"  - {d.page_content}")

Python 相关:
  - Python 教程
  - Python API

入门级别:
  - Python 教程
  - Go 基础


**术语：**
- **metadata 过滤** = 利用 Python 列表推导式按元数据字段筛选 Document
- **可变性** = Document 创建后可以修改 `page_content` 和 `metadata`

### 3. Document 在链中的应用

在实际应用中，文档从加载器产生、经分割器切分、被检索器召回，最终送入 LLM 进行生成。

In [5]:
# 模拟从向量检索中获取文档
retrieved_docs = [
    Document("Python 由 Guido van Rossum 创建于 1991 年", metadata={"source": "wiki"}),
    Document("Python 以简洁易读的语法著称", metadata={"source": "tutorial"}),
]

# 将文档组合成上下文，送入链中
def format_context(docs: list) -> dict:
    context = "\n".join(d.page_content for d in docs)
    return {"context": context, "question": "Python 的特点是什么？"}

chain = (
    RunnableLambda(format_context)
    | ChatPromptTemplate.from_template(
        "基于以下资料回答问题：\n\n{context}\n\n问题：{question}"
    )
    | llm | StrOutputParser()
)

result = chain.invoke(retrieved_docs)
print(result)

根据提供的资料，Python 的特点是**简洁易读的语法**。


**术语：**
- **RunnableLambda(format_context)** = 把格式化函数转成 Runnable，接收 Document 列表，返回 Prompt 所需的字典
- **检索增强生成（RAG）** = 先检索相关文档，再将文档内容作为上下文辅助 LLM 生成回答

```mermaid
%%{init: {'theme': 'dark'}}%%
graph LR
    A[文档加载] --> B[文本分割]
    B --> C[向量存储]
    C --> D[检索]
    D --> E[文档 + 提问]
    E --> F[LLM 生成回答]
```